## 1. Imports

In [1]:
import requests
import sqlite3
import json
import time
from datetime import datetime
import importlib
import configurar_db
import salvar_dados


# 2. Configurações base da API


In [2]:

## Aqui ele monta a URL da API do PNCP.
##	•	BASE_URL = parte principal do endereço
##	•	ENDPOINT = caminho específico do recurso que será consultado
##	•	URL_FINAL = junção dos dois


# --- Configurações Base ---
BASE_URL = "https://pncp.gov.br/api/consulta" #URL BASE
ENDPOINT = "/v1/contratacoes/proposta" #ENDPOINT p/ consultar contratações com período de recebimento em aberto

URL_FINAL = BASE_URL + ENDPOINT




## 3. Parâmetros da requisição

In [3]:
CODIGO_MODALIDADE = 6 # 6 = Dispensa de Licitação, conforme sua solicitação
DATA_FINAL = datetime.now().strftime('%Y%m%d') # Data de hoje no formato AAAAMMDD
print(DATA_FINAL)

20260313


## 4. Configuração dos parâmetros da consulta à API

Definição dos filtros utilizados na requisição ao PNCP:
- data final da consulta
- modalidade da contratação
- paginação da API
- tamanho da página

In [4]:

# Parâmetros obrigatórios e recomendados para a consulta
params = {
    'dataFinal': int(datetime.now().strftime('%Y%m%d')),
    'codigoModalidadeContratacao': 6,
    'pagina': 1,
    'tamanhoPagina': 50
}

# Nome do banco de dados SQLite onde os dados serão salvos
NOME_DB = "pncp_data.db"

# No loop, a atualização da página é feita da seguinte maneira:
# params['pagina'] = pagina_atual


In [5]:
# --- CÉLULA: IMPORTAÇÃO DA FUNÇÃO DE FETCH COM RETRY ---
# A função 'fetch_com_retry' foi movida para o arquivo 'fetch_retry.py'.

import importlib
import fetch_retry
importlib.reload(fetch_retry)  # Garante que alterações no arquivo sejam carregadas

from fetch_retry import fetch_com_retry

print("Módulo 'fetch_retry' carregado com sucesso.")


Módulo 'fetch_retry' carregado com sucesso.


In [6]:
teste = fetch_com_retry(URL_FINAL, params)

print(teste)

json_formatado = json.dumps(teste, indent=4, ensure_ascii=False)

print(json_formatado)


Código de resposta 200 (OK)
([{'tipoInstrumentoConvocatorioCodigo': 1, 'dataAtualizacao': '2026-03-02T11:42:07', 'orgaoEntidade': {'cnpj': '29138310000159', 'razaoSocial': 'MUNICIPIO DE MANGARATIBA', 'poderId': 'N', 'esferaId': 'M'}, 'anoCompra': 2025, 'sequencialCompra': 187, 'numeroCompra': '44', 'processo': 'PC 008290/2025', 'objetoCompra': 'Aquisição de equipamentos e materiais permanentes para a Atenção Especializada em Saúde, através de recurso financeiro proveniente da Proposta de Aquisição de Equipamentos/Material Permanente nº da Proposta: 12349.225000/1240 09. ', 'orgaoSubRogado': None, 'unidadeOrgao': {'ufNome': 'Rio de Janeiro', 'codigoUnidade': '1', 'ufSigla': 'RJ', 'municipioNome': 'Mangaratiba', 'nomeUnidade': 'GERAL', 'codigoIbge': '3302601'}, 'unidadeSubRogada': None, 'valorTotalHomologado': None, 'srp': False, 'dataInclusao': '2025-12-01T16:32:22', 'amparoLegal': {'codigo': 1, 'nome': 'Lei 14.133/2021, Art. 28, I', 'descricao': 'pregão: modalidade de licitação obrigat

In [7]:
# --- CÉLULA: IMPORTAÇÃO DA FUNÇÃO DE BUSCA PAGINADA ---
# A função 'buscar_dados_paginados' foi movida para o arquivo 'buscar_dados.py'.
# A função 'fetch_com_retry' foi movida para o arquivo 'fetch_retry.py'.

import importlib
import fetch_retry
import buscar_dados
importlib.reload(fetch_retry)   # Garante que alterações no arquivo sejam carregadas
importlib.reload(buscar_dados)  # Garante que alterações no arquivo sejam carregadas

from buscar_dados import buscar_dados_paginados

print("Módulos 'fetch_retry' e 'buscar_dados' carregados com sucesso.")


Módulos 'fetch_retry' e 'buscar_dados' carregados com sucesso.


In [8]:
buscar_dados_paginados(URL_FINAL, params, NOME_DB)


--- INICIANDO BUSCA PAGINADA ---

>>>> BUSCANDO PÁGINA: 1 de ???
Erro de conexão: HTTPSConnectionPool(host='pncp.gov.br', port=443): Read timed out. (read timeout=30). Tentando novamente em 1s...
Erro de conexão: HTTPSConnectionPool(host='pncp.gov.br', port=443): Read timed out. (read timeout=30). Tentando novamente em 1s...
Código de resposta 200 (OK)
   -> [DB] 0/50 registros salvos/atualizados na página 1.
Dados recebidos (Página 1/4, 50 registros):
[
    {
        "dataAtualizacao": "2026-03-02T11:42:07",
        "orgaoEntidade": {
            "cnpj": "29138310000159",
            "razaoSocial": "MUNICIPIO DE MANGARATIBA",
            "poderId": "N",
            "esferaId": "M"
        },
        "anoCompra": 2025,
        "sequencialCompra": 187,
        "numeroCompra": "44",
        "processo": "PC 008290/2025",
        "objetoCompra": "Aquisição de equipamentos e materiais permanentes para a Atenção Especializada em Saúde, através de recurso financeiro proveniente da Proposta de

True

In [9]:
# --- CÉLULA DE ML 1: IMPORTS (VERSÃO DEFINITIVA SIMPLIFICADA) ---
# Focamos em bibliotecas que não dependem de downloads externos que falham.

print("Carregando bibliotecas de ML (Versão Simplificada)...")

# --- Imports Essenciais ---
import pandas as pd
import numpy as np
import re
import joblib 

# --- Imports de NLTK: Usaremos  as stopwords ---
import nltk
from nltk.corpus import stopwords
# Removemos a importação de word_tokenize e PunktLanguageVars/PunktSentenceTokenizer

# --- Imports do Scikit-learn (sklearn) para o Pipeline de ML ---
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

print("Bibliotecas de ML importadas com sucesso.")

# --- Configuração de Stopwords ---
print(f"\n--- Preparando Stopwords em Português---")

try:
    # Carrega a lista de stopwords específicas do idioma 'portuguese'.
    lista_stopwords = stopwords.words('portuguese')
except LookupError:
     # Fallback seguro para o caso de falha TOTAL do NLTK
     print("ERRO: Stopwords não carregadas. Usando lista mínima de fallback.")
     lista_stopwords = ['a', 'o', 'e', 'de', 'para', 'com', 'um', 'uma']

# Converte a LISTA em um SET (conjunto) para busca rápida.
stop_words_pt = set(lista_stopwords)

print(f"Total de stopwords (SET) carregadas: {len(stop_words_pt)}")
print("--- Célula 10 concluída: Ferramentas prontas. Otimização final será feita na Célula 11. ---")

Carregando bibliotecas de ML (Versão Simplificada)...
Bibliotecas de ML importadas com sucesso.

--- Preparando Stopwords em Português---
Total de stopwords (SET) carregadas: 207
--- Célula 10 concluída: Ferramentas prontas. Otimização final será feita na Célula 11. ---
Bibliotecas de ML importadas com sucesso.

--- Preparando Stopwords em Português---
Total de stopwords (SET) carregadas: 207
--- Célula 10 concluída: Ferramentas prontas. Otimização final será feita na Célula 11. ---


In [10]:
# --- CÉLULA DE ML 2: IMPORTAÇÃO DA FUNÇÃO DE LIMPEZA DE TEXTO ---
# A função 'preprocessar_texto' foi movida para o arquivo 'preprocessar.py'.

import importlib
import preprocessar
importlib.reload(preprocessar)  # Garante que alterações no arquivo sejam carregadas

from preprocessar import preprocessar_texto, stop_words_pt

print(f"Módulo 'preprocessar' carregado com sucesso.")
print(f"Total de stopwords carregadas: {len(stop_words_pt)}")

# --- Teste Rápido ---
print("\n--- TESTANDO A FUNÇÃO 'preprocessar_texto' ---")
exemplo = "Pregão Eletrônico para Aquisição de 10 (dez) notebooks i7."
resultado = preprocessar_texto(exemplo)
print(f"  ORIGINAL:   '{exemplo}'")
print(f"  PROCESSADO: '{resultado}'")
print("--- Célula 2 concluída. Prossiga para a Célula 3 (Treinamento) ---")


Módulo 'preprocessar' carregado com sucesso.
Total de stopwords carregadas: 207

--- TESTANDO A FUNÇÃO 'preprocessar_texto' ---
  ORIGINAL:   'Pregão Eletrônico para Aquisição de 10 (dez) notebooks i7.'
  PROCESSADO: 'pregão eletrônico aquisição dez notebooks'
--- Célula 2 concluída. Prossiga para a Célula 3 (Treinamento) ---


In [11]:
# --- CÉLULA AUXILIAR: GERAR EXEMPLOS NEGATIVOS PARA O DATASET ---
# O erro "got 1 class" ocorre porque o dataset.csv só tem exemplos positivos (Relevante=1).
# Esta célula extrai licitações do banco e gera um CSV para você rotular manualmente.

import sqlite3
import pandas as pd
import json

NOME_DB = "pncp_data.db"
ARQUIVO_PARA_ROTULAR = "dataset_para_rotular.csv"
QUANTIDADE = 100  # Quantos exemplos extrair do banco

conn = sqlite3.connect(NOME_DB)
# Os dados estão armazenados como JSON na coluna 'dados_json'
rows = conn.execute(f"SELECT dados_json FROM contratacoes LIMIT {QUANTIDADE}").fetchall()
conn.close()

registros = []
for (dados_json,) in rows:
    data = json.loads(dados_json)
    registros.append({
        'pncp_id': data.get('numeroControlePNCP', ''),
        'Objeto': data.get('objetoCompra', '')
    })

df_banco = pd.DataFrame(registros)

# Adiciona coluna Relevante vazia para você preencher
df_banco['Relevante'] = ""

df_banco.to_csv(ARQUIVO_PARA_ROTULAR, index=False, encoding='utf-8-sig')

print(f"{len(df_banco)} licitações exportadas para '{ARQUIVO_PARA_ROTULAR}'")
print("\nInstrução:")
print("  1. Abra o arquivo 'dataset_para_rotular.csv'")
print("  2. Preencha a coluna 'Relevante' com 1 (relevante) ou 0 (irrelevante)")
print("  3. Copie as linhas rotuladas para o 'dataset.csv' existente")
print("  4. Certifique-se de ter ao menos 10 exemplos de cada classe (0 e 1)")
print("  5. Rode novamente a célula de Treinamento (ML 3)")
print(f"\nAmostra das licitações exportadas:")
print(df_banco[['Objeto', 'Relevante']].head(10).to_string(index=False))


100 licitações exportadas para 'dataset_para_rotular.csv'

Instrução:
  1. Abra o arquivo 'dataset_para_rotular.csv'
  2. Preencha a coluna 'Relevante' com 1 (relevante) ou 0 (irrelevante)
  3. Copie as linhas rotuladas para o 'dataset.csv' existente
  4. Certifique-se de ter ao menos 10 exemplos de cada classe (0 e 1)
  5. Rode novamente a célula de Treinamento (ML 3)

Amostra das licitações exportadas:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [12]:
# --- CÉLULA DE ML 3: TREINAMENTO DO MODELO (VERSÃO CORRIGIDA) ---
# O nome das colunas foi ajustado para corresponder ao que está no dataset.csv (Objeto e Relevante).

# --- ETAPA 1: DEFINIR NOMES DE ARQUIVOS ---

# Define o nome do arquivo CSV que você criou (o "gabarito").
NOME_ARQUIVO_DATASET = "dataset.csv"
# Define o nome do arquivo onde o modelo treinado será salvo.
NOME_ARQUIVO_MODELO = "modelo_relevancia.joblib"

print(f"--- INICIANDO TREINAMENTO ---")
print(f"Arquivo de Dataset (Gabarito): '{NOME_ARQUIVO_DATASET}'")
print(f"Arquivo de Saída (Modelo): '{NOME_ARQUIVO_MODELO}'")

# 'try...except' é um bloco de segurança. Ele tenta rodar o código.
# Se o 'dataset.csv' não for encontrado, ele pula para o bloco 'except' no final.
try:
    # --- ETAPA 2: CARREGAR O DATASET ---
    print(f"\n--- ETAPA 2: Carregando '{NOME_ARQUIVO_DATASET}' para um DataFrame pandas ---")
    
    # pd.read_csv lê o arquivo e o carrega na memória como um DataFrame (tabela).
    df_treino = pd.read_csv(NOME_ARQUIVO_DATASET)
    
    # Vamos inspecionar o DataFrame que acabamos de carregar:
    print(f"Tipo da variável 'df_treino': {type(df_treino)}")
    print(f"Formato (shape) do DataFrame (linhas, colunas): {df_treino.shape}")
    print(f"Total de {len(df_treino)} exemplos (licitações) carregados.")
    print("Amostra dos dados carregados (primeiras 5 linhas):")
    # .head() mostra as 5 primeiras linhas do DataFrame.
    print(df_treino.head())

    # --- CORREÇÃO APLICADA AQUI ---
    # Usando os nomes das colunas com a primeira letra MAIÚSCULA.
    print("\nContagem de valores na coluna 'Relevante' (para ver o balanceamento):")
    # .value_counts() mostra quantos '0's e '1's nós temos.
    print(df_treino['Relevante'].value_counts())
    
    # --- ETAPA 3: PRÉ-PROCESSAMENTO (LIMPEZA DE TEXTO) ---
    print(f"\n--- ETAPA 3: Aplicando pré-processamento (função 'preprocessar_texto') ---")
    
    # Aplicando em 'Objeto' (corrigido)
    df_treino['objeto_limpo'] = df_treino['Objeto'].apply(preprocessar_texto)
    
    # Vamos inspecionar o DataFrame DEPOIS da transformação:
    print("DataFrame atualizado com a coluna 'objeto_limpo'. Amostra:")
    # Acessa a coluna original 'Objeto' (corrigido)
    print(df_treino[['Objeto', 'objeto_limpo']].head())

    # --- ETAPA 4: SEPARAÇÃO DE FEATURES (X) E LABELS (y) ---
    print(f"\n--- ETAPA 4: Separando Features (X) e Labels (y) ---")
    
    # 'X' (Features): coluna 'objeto_limpo'.
    X = df_treino['objeto_limpo']
    
    # 'y' (Label/Target): coluna 'Relevante' (corrigido).
    y = df_treino['Relevante']
    
    # Inspecionando X e y:
    print(f"Tipo da variável 'X' (Features): {type(X)}")
    print(f"Tipo da variável 'y' (Labels): {type(y)}")
    print("\nAmostra de 'X' (o que o modelo vai ler):")
    print(X.head())
    print("\nAmostra de 'y' (o que o modelo vai tentar prever):")
    print(y.head())

    # --- ETAPA 5: DIVISÃO EM TREINO E TESTE ---
    print(f"\n--- ETAPA 5: Dividindo dados em Treino e Teste (80% / 20%) ---")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Inspecionando os 4 conjuntos de dados:
    print(f"Formato de 'X_train' (dados de treino): {X_train.shape}")
    print(f"Formato de 'y_train' (respostas de treino): {y_train.shape}")
    print(f"Formato de 'X_test' (dados de teste): {X_test.shape}")
    print(f"Formato de 'y_test' (respostas de teste): {y_test.shape}")

    # --- ETAPA 6: DEFINIÇÃO DO PIPELINE DE ML ---
    print(f"\n--- ETAPA 6: Definindo o Pipeline do Modelo ---")
    
    modelo_pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=1000, ngram_range=(1, 2))),
        ('clf', SGDClassifier(loss='hinge', random_state=42, max_iter=100))
    ])
    
    print("Objeto Pipeline criado com sucesso:")
    print(modelo_pipeline)

    # --- ETAPA 7: TREINAMENTO DO MODELO ---
    print(f"\n--- ETAPA 7: Treinando o modelo (método .fit()) ---")
    
    modelo_pipeline.fit(X_train, y_train)
    
    print("TREINAMENTO CONCLUÍDO.")

    # --- ETAPA 8: AVALIAÇÃO DO MODELO ---
    print(f"\n--- ETAPA 8: Avaliando o modelo nos dados de TESTE ---")
    
    y_pred = modelo_pipeline.predict(X_test)
    
    print(f"Tipo da variável 'y_pred' (Predições): {type(y_pred)}")
    print(f"Formato (shape) de 'y_pred': {y_pred.shape}")
    print("\nAmostra das predições vs. dados reais (primeiras 15):")
    print(f"  Predições (y_pred): {y_pred[:15]}")
    print(f"  Reais (y_test):     {y_test.values[:15]}")

    acuracia = accuracy_score(y_test, y_pred)
    print(f"\n==> Acurácia: {acuracia * 100:.2f}% <==")
    print("\nRelatório de Classificação Detalhado:")
    
    print(classification_report(y_test, y_pred, target_names=['Irrelevante (0)', 'Relevante (1)']))

    # --- ETAPA 9: SALVANDO O MODELO ---
    print(f"\n--- ETAPA 9: Salvando o modelo treinado em disco ---")
    
    joblib.dump(modelo_pipeline, NOME_ARQUIVO_MODELO)
    
    print(f"\nModelo salvo com sucesso como '{NOME_ARQUIVO_MODELO}'")
    print("--- Célula 12 (Treinamento) corrigida e concluída ---")

except FileNotFoundError:
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print(f"ERRO CRÍTICO: Arquivo '{NOME_ARQUIVO_DATASET}' não encontrado.")
    print(f"Por favor, crie este arquivo (PASSO 1) e rode esta célula novamente.")
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
except Exception as e:
    # Captura qualquer outro erro inesperado
    print(f"Ocorreu um erro inesperado: {e}")

--- INICIANDO TREINAMENTO ---
Arquivo de Dataset (Gabarito): 'dataset.csv'
Arquivo de Saída (Modelo): 'modelo_relevancia.joblib'

--- ETAPA 2: Carregando 'dataset.csv' para um DataFrame pandas ---
Tipo da variável 'df_treino': <class 'pandas.DataFrame'>
Formato (shape) do DataFrame (linhas, colunas): (74, 2)
Total de 74 exemplos (licitações) carregados.
Amostra dos dados carregados (primeiras 5 linhas):
                                              Objeto  Relevante
0  Contratação de serviços de calibração certific...          1
1  Registro de Preço para aquisição de hidrômetro...          1
2  Registro de Preço para fornecimento futuro e e...          1
3  Seleção Pública de Fornecedores para Aquisição...          1
4  Contratação de empresa especializada para aqui...          1

Contagem de valores na coluna 'Relevante' (para ver o balanceamento):
Relevante
1    74
Name: count, dtype: int64

--- ETAPA 3: Aplicando pré-processamento (função 'preprocessar_texto') ---
DataFrame atualiza

In [14]:
# --- CÉLULA DE ML 4: APLICANDO O MODELO NO BANCO DE DADOS ---
# Esta célula é a "fase de produção"
# 1. Carregar o modelo treinado (o .joblib) da Célula 3.
# 2. Conectar ao seu banco de dados SQLite (o .db) da coleta original.
# 3. Carregar TODOS os dados do banco para um DataFrame.
# 4. Aplicar a MESMA função de limpeza da Célula 2 nos dados do banco.
# 5. Usar o modelo carregado para PREVER quais são relevantes.
# 6. Salvar um novo CSV SÓ com as licitações relevantes.

import json

# --- ETAPA 1: DEFINIR NOMES DE ARQUIVOS ---
NOME_ARQUIVO_MODELO = "modelo_relevancia.joblib"
NOME_DB = "pncp_data.db"
ARQUIVO_SAIDA_FILTRADO = "licitacoes_filtradas.csv"

print(f"--- INICIANDO APLICAÇÃO DO MODELO ---")
print(f"Modelo a ser carregado: '{NOME_ARQUIVO_MODELO}'")
print(f"Banco de dados de origem: '{NOME_DB}'")
print(f"Arquivo de saída: '{ARQUIVO_SAIDA_FILTRADO}'")

try:
    # --- ETAPA 2: CARREGAR O MODELO TREINADO ---
    print(f"\n--- ETAPA 2: Carregando modelo '{NOME_ARQUIVO_MODELO}' do disco ---")
    modelo_carregado = joblib.load(NOME_ARQUIVO_MODELO)
    print("Modelo carregado com sucesso.")
    print(modelo_carregado)

    # --- ETAPA 3: CARREGAR DADOS DO BANCO SQLITE ---
    print(f"\n--- ETAPA 3: Conectando e lendo dados do SQLite '{NOME_DB}' ---")

    # Os dados estão armazenados como JSON na coluna 'dados_json'
    conn = sqlite3.connect(NOME_DB)
    rows = conn.execute("SELECT dados_json FROM contratacoes").fetchall()
    conn.close()
    print("Conexão com o banco fechada.")

    registros = []
    for (dados_json,) in rows:
        data = json.loads(dados_json)
        registros.append({
            'pncp_id': data.get('numeroControlePNCP', ''),
            'objeto_compra': data.get('objetoCompra', '')
        })

    df_db = pd.DataFrame(registros)

    print(f"\nDados carregados do banco com sucesso.")
    print(f"Formato (shape) do DataFrame (linhas, colunas): {df_db.shape}")
    print(f"Total de {len(df_db)} licitações carregadas do banco.")
    print("Amostra dos dados do banco (primeiras 5 linhas):")
    print(df_db.head())

    # --- ETAPA 4: APLICAÇÃO DO MODELO (SE HOUVER DADOS) ---
    if len(df_db) > 0:
        # --- 4a. PRÉ-PROCESSAMENTO (LIMPEZA) ---
        print(f"\n--- ETAPA 4a: Limpando os {len(df_db)} textos do banco de dados... ---")
        df_db['objeto_limpo'] = df_db['objeto_compra'].apply(preprocessar_texto)
        print("Limpeza concluída. Amostra:")
        print(df_db[['objeto_compra', 'objeto_limpo']].head())

        # --- 4b. PREDIÇÃO (CLASSIFICAÇÃO) ---
        print(f"\n--- ETAPA 4b: Aplicando o modelo para classificar {len(df_db)} itens... ---")
        predicoes = modelo_carregado.predict(df_db['objeto_limpo'])
        df_db['relevante_pred'] = predicoes

        print("Predição concluída.")
        print(f"Amostra das primeiras 15 predições: {predicoes[:15]}")
        print("\nContagem de predições (0s vs 1s):")
        print(df_db['relevante_pred'].value_counts())

        # --- 4c. FILTRAGEM ---
        print(f"\n--- ETAPA 4c: Filtrando apenas os relevantes (predição == 1) ---")
        df_relevante = df_db[df_db['relevante_pred'] == 1].copy()
        print(f"\nDe {len(df_db)} itens totais, {len(df_relevante)} foram classificados como RELEVANTES.")

        # --- 4d. SALVAMENTO DO RESULTADO ---
        print(f"\n--- ETAPA 4d: Salvando em '{ARQUIVO_SAIDA_FILTRADO}' ---")
        df_relevante.to_csv(ARQUIVO_SAIDA_FILTRADO, index=False, encoding='utf-8-sig')
        print(f"\nArquivo '{ARQUIVO_SAIDA_FILTRADO}' salvo com sucesso!")
        print("\nAmostra dos dados RELEVANTES encontrados (primeiras 10 linhas):")
        print(df_relevante[['pncp_id', 'objeto_compra']].head(10))

    else:
        print("ALERTA: O banco de dados está vazio. Rode o script de coleta primeiro.")

except FileNotFoundError:
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print(f"ERRO CRÍTICO: Modelo '{NOME_ARQUIVO_MODELO}' não encontrado.")
    print(f"Certifique-se de rodar a célula de Treinamento (ML 3) primeiro.")
    print(f"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")
    print(f"Verifique se a célula ML 2 ('preprocessar_texto') foi rodada.")

print("\n--- Célula ML 4 concluída ---")


--- INICIANDO APLICAÇÃO DO MODELO ---
Modelo a ser carregado: 'modelo_relevancia.joblib'
Banco de dados de origem: 'pncp_data.db'
Arquivo de saída: 'licitacoes_filtradas.csv'

--- ETAPA 2: Carregando modelo 'modelo_relevancia.joblib' do disco ---
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
ERRO CRÍTICO: Modelo 'modelo_relevancia.joblib' não encontrado.
Certifique-se de rodar a célula de Treinamento (ML 3) primeiro.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

--- Célula ML 4 concluída ---
